# 第107章 概率校准与Brier Score

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 22 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 类别不平衡与Top-K Lift  →  **本章任务：** 概率校准与Brier Score  →  **下一步：** K-Means客户分群实战
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景
**背景引入**：做预测时，模型除了给出“是 / 否”的答案，往往还会附带一个“有多确信”的概率，比如贷款是否违约、用户下个月是否复购。可如果模型说有 70% 的概率会违约，而实际违约的却连一半都不到，这份“把握”就华而不实，直接拿去做决策很容易出错。Brier Score 和概率校准，正是用来衡量并修正这种“预测概率与真实比例不符”的偏差，让模型上线前先过一道质检关。


## 本章目标

学完本章，你将能够：

- **理解**：理解「概率校准与Brier Score」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「概率校准与Brier Score」的关键输出指标。
- **迁移**：能把「概率校准与Brier Score」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：“模型说七成把握”到底靠不靠谱？如果嘴上说 70%、实际只有 60% 命中，那就是“说大话”。概率校准就是纠正这类“水分”，让模型报告的概率与真实比例尽量一致。校准要用没当过裁判的独立数据或内部交叉验证，否则就是自欺欺人。


- Brier=\(n^{-1}\sum_i(p_i-y_i)^2\)
- 校准概率 0.7 应约有 70% 正例（打个比方：说“七成把握”就得真有七成命中，这才叫说话算话；校准就是纠正那些嘴上80%、实际只有60%的“水分”。）
- sigmoid 稳健、isotonic 更灵活
- 校准必须使用独立数据或内部交叉验证


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | 参见本节示例 | 先明确样本、特征、目标和验证方式，再训练模型。 | AUC 高就认为概率可信 |
| 模型、公式与诊断 | `base.fit()`、`m.predict_proba()`、`rows.append()`、`pd.DataFrame()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 在测试集上拟合校准器 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-107 -->
### 数学推导｜Brier Score 衡量概率误差

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜逐样本计算概率平方误差。** $b_i=(p_i-y_i)^2$；预测越自信且越错误，惩罚越大。

**第 2 步｜对全部样本平均。** $BS=\sum_ib_i/n$。

**第 3 步｜按概率区间检查校准。** 若第 $g$ 个区间平均预测为 $\bar p_g$、实际正类率为 $\bar y_g$，校准误差的一种摘要是

$$
CE=\sum_g\frac{n_g}{n}(\bar p_g-\bar y_g)^2
$$

Brier 同时受校准与区分影响，因此还要配合校准曲线和排序指标。

**把上面的关系收束为本章计算式：**

$$
BS=\frac{1}{n}\sum_{i=1}^{n}(p_i-y_i)^2
$$

**符号解释：** $p_i$ 是预测概率，$y_i\in\{0,1\}$；分数越小越好。

**代码对应：** 结合校准曲线比较预测概率与实际发生率。

**使用边界：** 校准好不等于区分能力强；应与 ROC/PR 和业务阈值共同评估。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=96
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：示例先把数据切成了训练集与测试集。请把 `train_test_split` 里的切分比例改成 `test_size=0.3` 重新切分，感受训练 / 测试集行数如何变化，再看 `y_train` 里正例的比例，体会 `stratify=` 分层抽样对样本分布的作用。填好下方代码后运行自检，确认行数合计与总样本数一致。


In [ ]:
try:
    # 请在下方填写代码
    # 提示：
    #   1) 复制上面的 train_test_split，把切分比例改成 test_size=0.3
    #   2) 保存为新变量 X_train2、X_test2、y_train2、y_test2
    #   3) 观察并打印训练集、测试集行数与 y_train2 的正例比例
    X_train2, X_test2, y_train2, y_test2 = (None, None, None, None)
    # 在此补全你的切分与打印代码：
    #

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

base = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=3, random_state=96
)
raw = base.fit(X_train, y_train)
calibrated = CalibratedClassifierCV(base, method="sigmoid", cv=5).fit(
    X_train, y_train
)
rows = []
for name, m in [("raw", raw), ("calibrated", calibrated)]:
    p = m.predict_proba(X_test)[:, 1]
    rows.append([name, brier_score_loss(y_test, p), roc_auc_score(y_test, p)])
display(
    pd.DataFrame(rows, columns=["model", "Brier", "ROC_AUC"])
    .set_index("model")
    .round(4)
)
obs, forecast = calibration_curve(
    y_test, calibrated.predict_proba(X_test)[:, 1], n_bins=6
)
display(pd.DataFrame({"预测概率": forecast, "实际比例": obs}).round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- AUC 高就认为概率可信
- 在测试集上拟合校准器
- 小样本使用过细的校准分箱
- 部署后不监控概率漂移


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 107.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 107.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 107.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

检查分类概率是否可信，并使用校准曲线、Brier Score 和 CalibratedClassifierCV 改善概率。


### 你已经掌握

- 区分排序与校准
- 计算 Brier Score
- 读取校准分箱
- 使用交叉验证校准模型


### 需要注意

- AUC 高就认为概率可信
- 在测试集上拟合校准器
- 小样本使用过细的校准分箱
- 部署后不监控概率漂移


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
from sklearn.datasets import load_breast_cancer

# 恢复主数据集（前面的错误演示临时覆盖了 X、y）
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=96
)
print("训练集行数：", X_train2.shape[0], "测试集行数：", X_test2.shape[0])
print("训练集正例比例：", round(float(y_train2.sum()) / y_train2.size, 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_brier = {
    name: brier_score_loss(y_test, m.predict_proba(X_test)[:, 1])
    for name, m in [("raw", raw), ("calibrated", calibrated)]
}
print(practice_brier)
